<h1 style="color:DodgerBlue">Индивидальный проект</h1>

- группа: СМАРТб-25-1

- Решетова Арина


<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;

//  интерфейсы (для множественного наследования) 

public interface IRefundable
{
    void Refund(decimal amount);               // возврат денег
}

public interface INotifiable
{
    void SendNotification(string message);     // отправка уведомления
}

// Базовый класс

public class PaymentMethod
{
    // Атрибуты из задания
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }

    // Новые атрибуты
    public string Currency { get; set; }
    public bool IsActive { get; set; }
    public decimal TotalPaid { get; set; }

    public PaymentMethod(int id, string name, decimal minAmount)
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
        Currency = "RUB";
        IsActive = true;
        TotalPaid = 0;
    }

    // проверка минимальной суммы
    public virtual bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    // обработка платежа
    public virtual void ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"{MethodName}: способ оплаты отключен");
        }
        else if (CheckMinimumAmount(amount))
        {
            TotalPaid += amount;
            Console.WriteLine($"{MethodName}: платеж {amount} {Currency} выполнен");
        }
        else
        {
            Console.WriteLine($"{MethodName}: сумма {amount} меньше минимальной ({MinAmount})");
        }
    }

    // информация о способе оплаты
    public virtual string GetPaymentDetails()
    {
        return $"ID: {PaymentMethodId}, Способ: {MethodName}, Минимальная сумма: {MinAmount} {Currency}";
    }

    // Новые методы
    public void Activate()
    {
        IsActive = true;
        Console.WriteLine($"{MethodName}: способ оплаты включен");
    }

    public void Deactivate()
    {
        IsActive = false;
        Console.WriteLine($"{MethodName}: способ оплаты отключен");
    }

    public void ShowTotalPaid()
    {
        Console.WriteLine($"{MethodName}: всего оплачено {TotalPaid} {Currency}");
    }
}

//  Простое наследование 

// Онлайн-оплата
public class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; set; }
    public int SessionMinutes { get; set; }

    public OnlinePayment(int id, string name, decimal minAmount, string paymentUrl)
        : base(id, name, minAmount)
    {
        PaymentUrl = paymentUrl;
        SessionMinutes = 15;
    }

    // переопределяем: добавляем URL платежной системы
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Переход на платежную систему: {PaymentUrl}");
        base.ProcessPayment(amount);
    }

    // новый метод
    public void OpenSession()
    {
        Console.WriteLine($"Сессия оплаты открыта на {SessionMinutes} минут");
    }
}

// банковский перевод
public class BankTransfer : PaymentMethod
{
    public string BankData { get; set; }
    public decimal BankFee { get; set; }

    public BankTransfer(int id, string name, decimal minAmount, string bankData, decimal bankFee)
        : base(id, name, minAmount)
    {
        BankData = bankData;
        BankFee = bankFee;
    }

    // переопределяем: проверяем минимальную сумму с учетом комиссии банка
    public override bool CheckMinimumAmount(decimal amount)
    {
        return (amount - BankFee) >= MinAmount;
    }

    // новый метод
    public decimal GetAmountAfterFee(decimal amount)
    {
        return amount - BankFee;
    }
}

// наличные
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public string WorkingHours { get; set; }

    public CashPayment(int id, string name, decimal minAmount, string cashPickupPoint)
        : base(id, name, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
        WorkingHours = "09:00-21:00";
    }

    // переопределяем: показываем место выдачи наличных
    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", Место выдачи: {CashPickupPoint}";
    }

    // новый метод
    public void ShowWorkingHours()
    {
        Console.WriteLine($"Часы работы пункта: {WorkingHours}");
    }
}

//  сложное + множественное наследование 
// PaymentMethod -> OnlinePayment -> CardPayment
// и дополнительно интерфейсы IRefundable, INotifiable

public class CardPayment : OnlinePayment, IRefundable, INotifiable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }

    public CardPayment(int id, string name, decimal minAmount, string paymentUrl,
                       string cardNumber, string cardHolder)
        : base(id, name, minAmount, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
    }

    // скрываю номер карты и показываю послед 4 цифры
    public string GetMaskedNumber()
    {
        return "**** **** **** " + CardNumber.Substring(CardNumber.Length - 4);
    }

    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Оплата картой {GetMaskedNumber()}, владелец: {CardHolder}");
        base.ProcessPayment(amount);
        if (CheckMinimumAmount(amount) && IsActive)
        {
            SendNotification($"Платеж на {amount} {Currency} выполнен");
        }
    }

    // метод интерфейса IRefundable
    public void Refund(decimal amount)
    {
        if (amount <= TotalPaid)
        {
            TotalPaid -= amount;
            Console.WriteLine($"Возврат {amount} {Currency} на карту {GetMaskedNumber()}");
        }
        else
        {
            Console.WriteLine("Нельзя вернуть больше, чем было оплачено");
        }
    }

    // метод интерфейса INotifiable
    public void SendNotification(string message)
    {
        Console.WriteLine($"Уведомление: {message}");
    }
}

//  создание объектов и проверка 

PaymentMethod basePay = new PaymentMethod(1, "Универсальная оплата", 100m);
OnlinePayment online = new OnlinePayment(2, "Онлайн-оплата", 10m, "https://pay.example.com");
BankTransfer bank = new BankTransfer(3, "Банковский перевод", 100m, "Р/с 40817810000000000001", 30m);
CashPayment cash = new CashPayment(4, "Наличные", 50m, "Пункт выдачи №1, ул. Ленина, 5");
CardPayment card = new CardPayment(5, "Оплата картой", 10m, "https://pay.example.com/card",
                                   "4276123456789012", "ARINA RESHETOVA");

Console.WriteLine("--- Базовый класс ---");
basePay.ProcessPayment(50m);
basePay.ProcessPayment(500m);
Console.WriteLine(basePay.GetPaymentDetails());

Console.WriteLine("\n--- OnlinePayment ---");
online.OpenSession();
online.ProcessPayment(1500m);

Console.WriteLine("\n--- BankTransfer ---");
bank.ProcessPayment(120m);    // 120 - 30 = 90, это меньше 100
bank.ProcessPayment(5000m);
Console.WriteLine($"К зачислению с 5000: {bank.GetAmountAfterFee(5000m)}");

Console.WriteLine("\n--- CashPayment ---");
cash.ProcessPayment(2000m);
Console.WriteLine(cash.GetPaymentDetails());
cash.ShowWorkingHours();

Console.WriteLine("\n--- CardPayment ---");
card.ProcessPayment(2500m);
card.Refund(500m);
card.ShowTotalPaid();

Console.WriteLine("\n--- Полиморфизм ---");
PaymentMethod[] methods = { basePay, online, bank, cash, card };
foreach (PaymentMethod m in methods)
{
    Console.WriteLine(m.GetPaymentDetails());
}

Console.WriteLine("\n--- Отключение способа оплаты ---");
online.Deactivate();
online.ProcessPayment(300m);
online.Activate();

--- Базовый класс ---
Универсальная оплата: сумма 50 меньше минимальной (100)
Универсальная оплата: платеж 500 RUB выполнен
ID: 1, Способ: Универсальная оплата, Минимальная сумма: 100 RUB

--- OnlinePayment ---
Сессия оплаты открыта на 15 минут
Переход на платежную систему: https://pay.example.com
Онлайн-оплата: платеж 1500 RUB выполнен

--- BankTransfer ---
Банковский перевод: сумма 120 меньше минимальной (100)
Банковский перевод: платеж 5000 RUB выполнен
К зачислению с 5000: 4970

--- CashPayment ---
Наличные: платеж 2000 RUB выполнен
ID: 4, Способ: Наличные, Минимальная сумма: 50 RUB, Место выдачи: Пункт выдачи №1, ул. Ленина, 5
Часы работы пункта: 09:00-21:00

--- CardPayment ---
Оплата картой **** **** **** 9012, владелец: ARINA RESHETOVA
Переход на платежную систему: https://pay.example.com/card
Оплата картой: платеж 2500 RUB выполнен
Уведомление: Платеж на 2500 RUB выполнен
Возврат 500 RUB на карту **** **** **** 9012
Оплата картой: всего оплачено 2000 RUB

--- Полиморфизм ---
I